# Task 02C — decision-tilted SVGD falsification study

Select an **NVIDIA GPU** runtime before starting. GPU is strongly preferred for fresh float64 NumPyro NUTS and structural gradients; CPU is suitable only for preflight. `RUN_PREFLIGHT` and `RUN_FULL` are false, so **Run all is safe**. This notebook never authenticates to or pushes to GitHub.

The saved-teacher preflight needs the Task 02B full-output ZIP because raw signatures are intentionally not committed. Upload that ZIP in the guarded preflight cell. The full runner then fits independent fresh Task 02C teachers and repeats the gate before transport.

In [ ]:
REPO_URL = "https://github.com/PaulsonLab/energy-inference-bo.git"
REPO_REF = "main"  # Replace with the published Task 02C commit SHA for an archival run.
ACCELERATOR = "gpu"  # "gpu" strongly preferred; "cpu" is valid but slow.
RUN_PREFLIGHT = False
RUN_FULL = False


In [ ]:
import os, platform, sys
if sys.version_info[:2] not in {(3, 11), (3, 12)}:
    raise RuntimeError(f"Unsupported Python {platform.python_version()}; use Colab Python 3.11 or 3.12.")
if ACCELERATOR not in {"cpu", "gpu"}:
    raise ValueError("ACCELERATOR must be 'cpu' or 'gpu'.")
os.environ["JAX_PLATFORMS"] = "cuda" if ACCELERATOR == "gpu" else "cpu"
os.environ["JAX_ENABLE_X64"] = "true"
os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false"
print("Python", platform.python_version(), "requested accelerator", ACCELERATOR, "JAX x64 enabled")


In [ ]:
from pathlib import Path
import subprocess
REPO_DIR = Path("/content/energy-inference-bo")
if REPO_DIR.exists():
    raise RuntimeError(f"{REPO_DIR} exists; restart the runtime for a clean archival run.")
if ACCELERATOR == "gpu":
    subprocess.run(["nvidia-smi"], check=True)
subprocess.run(["git", "clone", "--filter=blob:none", REPO_URL, str(REPO_DIR)], check=True)
# Explicitly fetch the requested branch or archival SHA: blob-filtered clones may not have every commit object locally.
subprocess.run(["git", "fetch", "--depth=1", "origin", REPO_REF], cwd=REPO_DIR, check=True)
subprocess.run(["git", "checkout", "--detach", "FETCH_HEAD"], cwd=REPO_DIR, check=True)
GIT_SHA = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=REPO_DIR, text=True).strip()
print("Checked out", GIT_SHA)


In [ ]:
subprocess.run([sys.executable, "-m", "pip", "install", "-r", str(REPO_DIR / "requirements.txt")], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "pytest==9.1.1"], check=True)
if ACCELERATOR == "gpu":
    subprocess.run([sys.executable, "-m", "pip", "install", "jax[cuda12]==0.9.2"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-e", str(REPO_DIR)], check=True)


In [ ]:
import importlib.metadata as metadata
import jax
jax.config.update("jax_enable_x64", True)
devices = jax.devices()
if not jax.config.jax_enable_x64:
    raise RuntimeError("Task 02C requires JAX float64.")
if ACCELERATOR == "gpu" and (jax.default_backend() != "gpu" or not any(d.platform == "gpu" for d in devices)):
    raise RuntimeError(f"GPU requested but JAX reports backend={jax.default_backend()}, devices={devices}")
if ACCELERATOR == "cpu" and jax.default_backend() != "cpu":
    raise RuntimeError(f"CPU requested but JAX reports {jax.default_backend()}")
print("JAX", jax.__version__, jax.default_backend(), devices, "x64=", jax.config.jax_enable_x64)
for package in ["energy-inference-bo", "torch", "botorch", "gpytorch", "numpyro"]:
    print(package, metadata.version(package))
subprocess.run([sys.executable, "-m", "pytest", "-q"], cwd=REPO_DIR, check=True)


## Mandatory saved-teacher preflight

Set `RUN_PREFLIGHT=True`, run this cell, and upload the **Task 02B full outputs ZIP** when prompted. It must contain `artifacts/task02b/full/signatures/seed*_n*.npz`. The cell extracts only into this disposable Colab checkout and runs no NUTS.

In [ ]:
PREFLIGHT_OK = False
if not RUN_PREFLIGHT:
    raise RuntimeError("Preflight disabled. Set RUN_PREFLIGHT=True and rerun this cell.")
from google.colab import files
import json, zipfile
uploaded = files.upload()
if len(uploaded) != 1:
    raise RuntimeError("Upload exactly one Task 02B full-output ZIP.")
zip_name, zip_bytes = next(iter(uploaded.items()))
zip_path = Path("/content") / zip_name
zip_path.write_bytes(zip_bytes)
with zipfile.ZipFile(zip_path) as archive:
    if any(Path(name).is_absolute() or '..' in Path(name).parts for name in archive.namelist()):
        raise RuntimeError("Unsafe path in uploaded ZIP.")
    archive.extractall(REPO_DIR)
SIGNATURE_DIR = REPO_DIR / "artifacts/task02b/full/signatures"
required = [SIGNATURE_DIR / f"seed{s}_n{n}.npz" for s in range(3) for n in (16, 40)]
if not all(path.exists() for path in required):
    raise RuntimeError(f"ZIP lacks required raw signatures under {SIGNATURE_DIR}")
PREFLIGHT_DIR = REPO_DIR / "artifacts/task02c/preflight"
PREFLIGHT_COMMAND = [sys.executable, "-m", "energy_bo.experiments.run_task02c", "--profile", "preflight", "--signature-dir", str(SIGNATURE_DIR), "--output-dir", str(PREFLIGHT_DIR)]
subprocess.run(PREFLIGHT_COMMAND, cwd=REPO_DIR, check=True, env=os.environ.copy())
preflight = json.loads((PREFLIGHT_DIR / "task02c_preflight.json").read_text())
PREFLIGHT_OK = bool(preflight["passed"])
print(json.dumps({k: preflight[k] for k in ["passed", "maxima", "potential_validation", "beta1_ess_fraction_min", "beta1_ess_fraction_max"]}, indent=2))
if not PREFLIGHT_OK:
    raise RuntimeError("Teacher preflight failed; do not run transport.")


## Explicit full run

Proceed only after `PREFLIGHT_OK` is true. Set `RUN_FULL=True` and run this cell. The six fresh NUTS teacher files are saved incrementally, and every fresh teacher must pass its own envelope/potential gate before transport begins for that case.

In [ ]:
if not RUN_FULL:
    raise RuntimeError("Full run disabled. Set RUN_FULL=True only after reviewing preflight.")
if not PREFLIGHT_OK:
    raise RuntimeError("Full run blocked: the mandatory preflight has not passed in this runtime.")
OUTPUT_DIR = REPO_DIR / "artifacts/task02c/full"
FULL_COMMAND = [sys.executable, "-m", "energy_bo.experiments.run_task02c", "--profile", "full", "--signature-dir", str(SIGNATURE_DIR), "--output-dir", str(OUTPUT_DIR)]
subprocess.run(FULL_COMMAND, cwd=REPO_DIR, check=True, env=os.environ.copy())


In [ ]:
import shutil
manifest = {
    "task": "task02c", "profile": "full", "git_sha": GIT_SHA,
    "python": platform.python_version(), "accelerator": ACCELERATOR,
    "jax_backend": jax.default_backend(), "jax_devices": [str(d) for d in devices], "jax_x64": bool(jax.config.jax_enable_x64),
    "packages": {name: metadata.version(name) for name in ["jax", "jaxlib", "numpyro", "torch", "botorch", "gpytorch"]},
    "commands": [PREFLIGHT_COMMAND, FULL_COMMAND],
}
(OUTPUT_DIR / "colab_manifest.json").write_text(json.dumps(manifest, indent=2) + "\n")
archive = Path(shutil.make_archive("/content/task02c_full_outputs", "zip", root_dir=REPO_DIR, base_dir="artifacts/task02c"))
print(archive, archive.stat().st_size, "bytes")
files.download(str(archive))


## What to do with the ZIP

Unzip locally into `artifacts/task02c/`. Review the full summary, methods table, fresh-teacher preflight, configuration, and manifest. Commit only reviewed aggregate evidence under `results/task02c/full/`; keep raw `teachers/*.npz` and detailed traces ignored. Uploading/downloading does not update GitHub—promotion is a separate local Git commit.